# 07_merge_oecd_quality

**Objective:** Merge the seven OECD patent-quality indicators onto the patent sample, converting BigQuery patent numbers to OECD keys and aggregating each indicator to one value per family (within-family mean).

**Inputs:** OECD INDIC 7z archives; BigQuery family/publication results; `novelty_features.parquet`; `lens_to_familyid.parquet`.

**Outputs:** `family_oecd_quality.parquet`; `novelty_with_oecd_quality.parquet`.

In [ ]:
# Imports
import re
import subprocess
from pathlib import Path

import pandas as pd

## Paths, archives and indicator column lists

In [ ]:
# Paths, archives, and indicator column lists
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
HERE = PROC
ARCHIVE_DIR = RAW / "oecd"
EPO_ARCHIVE = ARCHIVE_DIR / "202602_OECD_PATENT_QUALITY_EPO_INDIC.7z"
USPTO_ARCHIVE = ARCHIVE_DIR / "202602_OECD_PATENT_QUALITY_USPTO_INDIC.7z"
EPO_INNER = "202602_OECD_PATENT_QUALITY_EPO_INDIC.txt"
USPTO_INNER = "202602_OECD_PATENT_QUALITY_USPTO_INDIC.txt"
BQ_CSV = RAW / "bigquery" / "bq-results-20260511-101758-1778494803796.csv"

EP_INDICATORS = [
    "filing", "tech_field", "patent_scope", "family_size", "grant_lag",
    "bwd_cits", "npl_cits", "claims", "claims_bwd",
    "fwd_cits5", "fwd_cits5_xy", "fwd_cits7", "fwd_cits7_xy",
    "breakthrough", "breakthrough_xy",
    "generality", "originality", "radicalness", "renewal",
    "quality_index_4", "quality_index_6",
]
US_INDICATORS = [
    "filing", "tech_field", "patent_scope", "family_size", "grant_lag",
    "bwd_cits", "npl_cits", "claims", "claims_bwd",
    "fwd_cits5", "fwd_cits7",
    "breakthrough", "generality", "originality", "radicalness", "renewal",
    "quality_index_4", "quality_index_6",
]

## Key-conversion helpers (BQ application/publication numbers -> OECD keys)

In [ ]:
# Convert BigQuery application/publication numbers to OECD keys
def ep_app_to_oecd(app_num: str) -> str | None:
    """BQ 'EP-YYNNNNNN-A' -> OECD 'EPYYYY0NNNNNN'. Year from app-num prefix
    (78-99 -> 19YY, 00-77 -> 20YY); filing_date is unreliable for divisionals."""
    if not isinstance(app_num, str):
        return None
    m = re.match(r"^EP-(\d{2})(\d+)-", app_num)
    if not m:
        return None
    yy, serial = int(m.group(1)), m.group(2)
    year = 1900 + yy if yy >= 78 else 2000 + yy
    return f"EP{year}{serial.zfill(7)}"


def us_pub_to_oecd(pub_num: str, grant_date: str) -> str | None:
    """BQ 'US-NNNNNNNN-B[12]' grant -> OECD 'USNNNNNNNN' (8-digit zero-padded).
    Only granted utility patents (B1/B2) are in OECD's USPTO pub_nbr."""
    if not isinstance(pub_num, str) or not isinstance(grant_date, str) or grant_date == "0":
        return None
    m = re.match(r"^US-(\d+)-(B\d?)$", pub_num)
    if not m:
        return None
    return f"US{m.group(1).zfill(8)}"

## Build BQ -> OECD family key maps

In [ ]:
# Build BigQuery-to-OECD family key maps
def build_key_maps() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read the BQ family-publication CSV and emit:
       ep_map: family_id, oecd_app_nbr
       us_map: family_id, oecd_pub_nbr"""
    bq = pd.read_csv(BQ_CSV, dtype=str)

    ep = bq[bq.country_code == "EP"].copy()
    ep["oecd_app_nbr"] = ep["application_number"].apply(ep_app_to_oecd)
    ep_map = (
        ep.dropna(subset=["oecd_app_nbr"])[["family_id", "oecd_app_nbr"]]
          .drop_duplicates()
    )

    us = bq[bq.country_code == "US"].copy()
    us["oecd_pub_nbr"] = us.apply(
        lambda r: us_pub_to_oecd(r["publication_number"], r["grant_date"]),
        axis=1,
    )
    us_map = (
        us.dropna(subset=["oecd_pub_nbr"])[["family_id", "oecd_pub_nbr"]]
          .drop_duplicates()
    )

    print(f"EP key map: {len(ep_map):,} rows, {ep_map.family_id.nunique():,} families")
    print(f"US key map: {len(us_map):,} rows, {us_map.family_id.nunique():,} families")
    return ep_map, us_map

## Stream a 7z-compressed OECD INDIC file, keeping matching keys

In [ ]:
# Stream a 7z-compressed OECD INDIC file, keeping matching keys
def stream_oecd(archive: Path, inner: str, key_column: str, keys: set[str],
                indicator_cols: list[str], chunksize: int = 500_000) -> pd.DataFrame:
    """Stream a 7z-compressed OECD INDIC file through 7zz -> pandas, keeping
    only rows whose key_column is in `keys`."""
    proc = subprocess.Popen(
        ["7zz", "x", "-so", str(archive), inner],
        stdout=subprocess.PIPE,
    )
    use_cols = [key_column] + indicator_cols
    matched_chunks = []
    rows_scanned = 0
    for chunk in pd.read_csv(
        proc.stdout, sep="|", dtype=str, usecols=use_cols, chunksize=chunksize,
        on_bad_lines="warn",
    ):
        rows_scanned += len(chunk)
        m = chunk[chunk[key_column].isin(keys)]
        if len(m):
            matched_chunks.append(m)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"7zz failed extracting {inner}")
    matched = pd.concat(matched_chunks, ignore_index=True) if matched_chunks else pd.DataFrame(columns=use_cols)
    print(f"  {inner}: scanned {rows_scanned:,} rows, matched {len(matched):,}")
    for c in indicator_cols:
        matched[c] = pd.to_numeric(matched[c], errors="coerce")
    return matched

## Aggregate OECD indicators to one row per family

In [ ]:
# Aggregate OECD indicators to one row per family
def aggregate_per_family(oecd_df: pd.DataFrame, key_map: pd.DataFrame,
                        key_column: str, indicator_cols: list[str],
                        prefix: str) -> pd.DataFrame:
    """Join oecd_df rows to family_id via key_map and aggregate one row per family.
    The five model indicators (bwd_cits, npl_cits, claims, family_size, patent_scope) are
    aggregated by MEAN across family members, so the family value reflects the typical
    member rather than the single largest. Forward-looking / unused indicators use max."""
    joined = oecd_df.merge(key_map, left_on=key_column, right_on=key_map.columns[1], how="left")
    joined = joined.dropna(subset=["family_id"])

    max_cols = [c for c in indicator_cols
                if c in {"fwd_cits5", "fwd_cits5_xy", "fwd_cits7", "fwd_cits7_xy",
                         "claims_bwd", "breakthrough", "breakthrough_xy"}]
    mean_cols = [c for c in indicator_cols
                 if c in {"bwd_cits", "npl_cits", "claims", "family_size", "patent_scope",
                          "generality", "originality", "radicalness",
                          "quality_index_4", "quality_index_6", "renewal",
                          "grant_lag"}]

    agg = joined.groupby("family_id").agg(
        {**{c: "max" for c in max_cols}, **{c: "mean" for c in mean_cols}}
    )
    n_apps = joined.groupby("family_id").size().rename(f"{prefix}n_apps")
    agg = agg.add_prefix(prefix).join(n_apps)
    return agg.reset_index()

## Main pipeline: build maps, stream, aggregate, join, correlate

In [ ]:
# Main pipeline: build maps, stream, aggregate, join, correlate
def main():
    print("=" * 60)
    print("1. Build BQ -> OECD key maps")
    print("=" * 60)
    ep_map, us_map = build_key_maps()
    ep_keys = set(ep_map["oecd_app_nbr"])
    us_keys = set(us_map["oecd_pub_nbr"])

    print()
    print("=" * 60)
    print("2. Stream OECD EPO INDIC (this takes ~1 min)")
    print("=" * 60)
    ep_indic = stream_oecd(EPO_ARCHIVE, EPO_INNER, "app_nbr", ep_keys, EP_INDICATORS)

    print()
    print("=" * 60)
    print("3. Stream OECD USPTO INDIC (this takes ~3 min — 1.4 GB)")
    print("=" * 60)
    us_indic = stream_oecd(USPTO_ARCHIVE, USPTO_INNER, "pub_nbr", us_keys, US_INDICATORS)

    print()
    print("=" * 60)
    print("4. Aggregate per family")
    print("=" * 60)
    ep_agg = aggregate_per_family(ep_indic, ep_map, "app_nbr", EP_INDICATORS, "ep_")
    us_agg = aggregate_per_family(us_indic, us_map, "pub_nbr", US_INDICATORS, "us_")
    print(f"  EP aggregated: {len(ep_agg):,} families")
    print(f"  US aggregated: {len(us_agg):,} families")

    family_oecd = ep_agg.merge(us_agg, on="family_id", how="outer")
    family_oecd.to_parquet(HERE / "family_oecd_quality.parquet")
    print(f"  -> family_oecd_quality.parquet ({len(family_oecd):,} families)")

    print()
    print("=" * 60)
    print("5. Join to novelty_features")
    print("=" * 60)
    nov = pd.read_parquet(HERE / "novelty_features.parquet")
    fam = pd.read_parquet(HERE / "lens_to_familyid.parquet")
    nov_fam = nov.merge(fam[["lens_id", "docdb_family_id"]], on="lens_id", how="left")
    nov_fam["family_id"] = (
        nov_fam["docdb_family_id"].astype("Int64").astype(str)
    )
    result = nov_fam.merge(family_oecd, on="family_id", how="left")
    result.to_parquet(HERE / "novelty_with_oecd_quality.parquet")
    print(f"  -> novelty_with_oecd_quality.parquet ({len(result):,} rows)")
    print(f"  Rows with any OECD indicator: {result['ep_fwd_cits5'].notna().sum() + result['us_fwd_cits5'].notna().sum():,}")
    print(f"  Rows with EP fwd_cits5_xy:  {result['ep_fwd_cits5_xy'].notna().sum():,}")
    print(f"  Rows with US fwd_cits5:     {result['us_fwd_cits5'].notna().sum():,}")

    print()
    print("=" * 60)
    print("6. Quick novelty-vs-citations correlations")
    print("=" * 60)
    pairs = [
        ("novelty_mean", "ep_fwd_cits5_xy"),
        ("novelty_min",  "ep_fwd_cits5_xy"),
        ("novelty_top10","ep_fwd_cits5_xy"),
        ("novelty_mean", "ep_fwd_cits5"),
        ("novelty_mean", "us_fwd_cits5"),
        ("novelty_mean", "ep_quality_index_6"),
    ]
    for a, b in pairs:
        sub = result[[a, b]].dropna()
        if len(sub) < 100:
            print(f"  {a:14s} vs {b:18s}: n={len(sub)} (too few)")
            continue
        pearson  = sub.corr(method="pearson").iloc[0, 1]
        spearman = sub.corr(method="spearman").iloc[0, 1]
        print(f"  {a:14s} vs {b:18s}: n={len(sub):>6,}  pearson={pearson:+.3f}  spearman={spearman:+.3f}")

In [ ]:
# Entry point
if __name__ == "__main__":
    main()